# Demande conditionnelle des facteurs avec contrôles en 2022

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

## Chargement des données 

In [2]:
finess = pd.read_excel("finess.xlsx")
sygen = pd.read_csv("SAE/2022/SYGEN_2022r.csv", sep=";", encoding="latin-1")
urg = pd.read_csv("SAE/2022/URGENCES2_2022r.csv", sep=";", encoding="latin-1")
filtre = pd.read_csv("SAE/2022/FILTRE_2022r.csv", sep=";", decimal = ",", encoding="latin-1")
hd = pd.read_csv("Hospidiag/hd2022.csv", sep=";", decimal = ",", encoding="latin-1")

## Nettoyage des données

In [3]:
data = sygen

In [4]:
finess = finess[["FINESS","Raison sociale","Statut Juridique"]]
finess = finess.rename(columns={"FINESS":"FI","Raison sociale":"RS","Statut Juridique":"Statut"})
data = data.merge(finess, how="left", on = "FI")

In [5]:
urg = urg[['FI','PASSU']]
data = data.merge(urg, on="FI", how="left")

In [6]:
hd = hd.rename(columns={"finess":"FI_EJ"})
data = data.merge(hd, how="left", on="FI_EJ")

In [7]:
filtre = filtre[["FI","HEB_MED","HEB_CHIR","HEB_PERINAT","HEB_PSY","HEB_SSR", "HEB_SLD"]]
data = data.merge(filtre, how="left", on="FI")

## Agrégation des personnels 

In [8]:
data['MED'] = data['EFFSAL_TOT'] + data['EFFLIB_TOT']
data['ADMIN'] = data['EFF_DIR'] + data['EFF_AUTADM']
data['IDE'] = data['EFF_INFSANSSPE']
data['AID'] = data['EFF_AID']

## Agrégation des séances et des séjours en psychiatrie

In [9]:
data['SEANCES'] = data['SEAN_HEMO_CENTRE'] + data['SEAN_CHIMIO'] + data['SEAN_RADIO']
data['PSY'] = data['SEJ_HTP_TOT'] + data['VEN_HDJ_TOT'] + data['VEN_HDN_TOT']
data['SSR'] = data['SEJHC_SSR']
data['SLD'] = data['ENT']

## Passage des variables en log

In [10]:
data["lURG"] = np.where(data["PASSU"] > 0, np.log(data["PASSU"]), 0)
data["lMCO_AMB"] = np.where(data["SEJHP_MCO"] > 0, np.log(data["SEJHP_MCO"]), 0)
data["lMCO_COMP"] = np.where(data["SEJHC_MCO"] > 0, np.log(data["SEJHC_MCO"]), 0)
data["lPSY"] = np.where(data["PSY"] > 0, np.log(data["PSY"]), 0)
data["lSEANCES"] = np.where(data["SEANCES"] > 0, np.log(data["SEANCES"]), 0)
data["lSSR"] = np.where(data["SSR"] > 0, np.log(data["SSR"]), 0)
data["lSLD"] = np.where(data["SLD"] > 0, np.log(data["SLD"]), 0)

data["lMED"] = np.where(data["MED"] > 0, np.log(data["MED"]), 0)
data["lIDE"] = np.where(data["IDE"] > 0, np.log(data["IDE"]), 0)
data["lAID"] = np.where(data["AID"] > 0, np.log(data["AID"]), 0)
data["lADMIN"] = np.where(data["ADMIN"] > 0, np.log(data["ADMIN"]), 0)

c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufun

In [11]:
np.isinf(data['lMED']).sum()

np.int64(0)

## Création des variables de statut

On veut 4 statuts : CHU, petit hôpital public, privé lucratif et privé non lucratif. On a déjà public, privé lucratif et privé non lucratif. On veut donc distinguer petit hôpital public et CHU parmi les hôpitaux publics. Les petits hôpitaux publics (publics mais non CHU) seront notre référence. 

In [12]:
chu = " CHU|CHU "
CHU_data = data[data['RS'].str.contains(chu, na=False)]
data['CHU'] = data['FI'].isin(CHU_data['FI']).astype(int)

In [13]:
data['PL'] = (data['Statut'] == 'Privé lucratif').astype(int)
data['PNL'] = (data['Statut'] == 'Privé non lucratif').astype(int)

## Demande conditionnelle de facteurs avec contrôles : médecins

In [14]:
outputs = ["lURG", "lMCO_AMB", "lMCO_COMP", "lSSR", "lPSY","lSEANCES", "lSLD"]
controls = ["PNL", "PL", "CHU","HEB_MED","HEB_CHIR","HEB_PERINAT","HEB_PSY","HEB_SSR", "HEB_SLD", "A14"]
inputs = ["lMED", "lIDE", "lAID", "lADMIN"]

In [22]:
for con in controls:
    print(data[con].isna().sum()/data.shape[0])

0.0
0.0
0.0
0.00012322858903265557
0.00012322858903265557
0.00012322858903265557
0.00012322858903265557
0.00012322858903265557
0.00012322858903265557
0.7826247689463955


In [33]:
hd_test = hd[['FI_EJ','A14']]
hd_test['A14'].isna().sum()/hd_test.shape[0]

np.float64(0.47315436241610737)

In [35]:
results = {}
dict = {}

for var in outputs:
    data[f"{var}_PNL"] = data[var] * data["PNL"]
    data[f"{var}_PL"] = data[var] * data["PL"]
    data[f"{var}_CHU"] = data[var] * data["CHU"]


for inp in inputs:
    X = data[outputs + controls + [f"{v}_PNL" for v in outputs] + [f"{v}_PL" for v in outputs] + [f"{v}_CHU" for v in outputs]]
    X = sm.add_constant(X)
    y = data[inp]
    
    model = sm.OLS(y, X, missing='drop').fit(cov_type="HC1")  # erreurs robustes
    results[inp] = model
    
    print("\n" + "="*60)
    print(f"Demande conditionnelle pour : {inp}")
    print(model.summary())
    
    print(inp)
    for c in ["PL","PNL","CHU"]:
        coef = model.params[c]
        pct = (np.exp(coef) - 1) * 100
        print(f"{c} → différence d'utilisation : {pct:.2f}%")
        dict[(inp,c)] = pct


Demande conditionnelle pour : lMED
                            OLS Regression Results                            
Dep. Variable:                   lMED   R-squared:                       0.248
Model:                            OLS   Adj. R-squared:                  0.238
Method:                 Least Squares   F-statistic:                     26.42
Date:                Fri, 13 Feb 2026   Prob (F-statistic):           8.77e-93
Time:                        16:22:35   Log-Likelihood:                -3529.9
No. Observations:                1764   AIC:                             7106.
Df Residuals:                    1741   BIC:                             7232.
Df Model:                          22                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const     

c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 38, but rank is 22
  warnings.warn('covariance of constraints does not have full '
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 38, but rank is 22
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:                   lIDE   R-squared:                       0.774
Model:                            OLS   Adj. R-squared:                  0.771
Method:                 Least Squares   F-statistic:                     336.4
Date:                Fri, 13 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:22:35   Log-Likelihood:                -2414.8
No. Observations:                1764   AIC:                             4876.
Df Residuals:                    1741   BIC:                             5002.
Df Model:                          22                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             1.7102      0.094     18.189

c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 38, but rank is 22
  warnings.warn('covariance of constraints does not have full '
c:\Users\roman\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 38, but rank is 22
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:                   lAID   R-squared:                       0.735
Model:                            OLS   Adj. R-squared:                  0.732
Method:                 Least Squares   F-statistic:                     219.1
Date:                Fri, 13 Feb 2026   Prob (F-statistic):               0.00
Time:                        16:22:35   Log-Likelihood:                -2521.5
No. Observations:                1764   AIC:                             5089.
Df Residuals:                    1741   BIC:                             5215.
Df Model:                          22                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             1.6379      0.093     17.611

In [ ]:
#OK : ajouter indic  HEB_MED, HEB_CHIR,HEB_PERINAT,HEB_PSY,HEB_SSR du fichier filtre (ils sont déjà en indic)

#OK: voir comment traiter USLD car beaucoup de données manquantes : si on l'inclut, ajouter indic ULSD HEB_SLD 
# -> ajouté USLD et indic malgré le manque de données

#OK: rajouter log output x statut juridique 

#OK: rajouter distiction CHU/petit hôpital public, mettre petits hopitaux publics en réf -> comment est ce qu'on fait vu 
#qu'on est au niveau du finess géographique : est ce que les unités appartenant à un CHU sont considérées comme CHU ? je dirais oui

#gros problème : avec le manque de données en USLD et sur l'indice de sévérité, on ne garde que 1764 observations / 8115...

#ajouter durée de séjour

#est ce qu'on distingue activité psychiatrique à temps complet et à temps partiel ? pour le moment agrégés dans PSY, 
#est ce que le temps partiel correspond à l'ambulatoire ? dans ce cas autant distinguer comme pour MCO

In [ ]:
data['SEJHC_SSR'].count() / data.shape[0]


In [ ]:
data['ENT'].count() / data.shape[0]